# Submitting Qiskit Circuits to Azure Quantum with the QDK

This notebook demonstrates how to run Qiskit `QuantumCircuit` jobs on Azure Quantum using `AzureQuantumProvider` from the QDK. It also shows how to handle parameterized circuits by binding parameters prior to execution.

The workflow demonstrated here:

1. Build a Qiskit `QuantumCircuit`.
2. Reference an existing Azure Quantum workspace with `qdk.azure.Workspace`.
3. Instantiate `AzureQuantumProvider(workspace)` and browse available backends (`provider.backends()`).
4. Pick a backend and call `backend.run(circuit, shots=...)` and fetch results (`job.result().get_counts()`).

## Prerequisites

This notebook assumes the `qdk`, Azure Quantum integration, and Qiskit packages are installed. You can install everything (including the Azure + Qiskit extras) with:

In [ ]:
%pip install "qdk[azure,qiskit]"

This installs:
- The base qdk package (compiler, OpenQASM/QIR tooling)
- Azure Quantum client dependencies for submission
- Qiskit for circuit construction

After installing, restart the notebook kernel if it was already running. You can verify installation with:

In [ ]:
import qiskit, qdk, qdk.azure  # should import without errors

## Build a simple Qiskit circuit

We start with a Bell-state circuit — a Hadamard on qubit 0 followed by a CNOT — which produces the entangled state $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$. After constructing the circuit we'll submit it to an Azure Quantum target.

In [ ]:
from qiskit import QuantumCircuit

q = QuantumCircuit(2, 2)
q.h(0)
q.cx(0, 1)
q.measure([0, 1], [0, 1])

circuit = q
circuit.draw(output="text")

## Configure Azure Quantum workspace connection

To connect to an Azure workspace replace the following variables with your own values.

In [ ]:
subscription_id = 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
resource_group = 'myresourcegroup'
workspace_name = 'myworkspace'
location = 'westus'

## Submit the circuit to Azure Quantum

`AzureQuantumProvider` exposes Azure Quantum targets as Qiskit backends. When you call `provider.get_backend(target_name)`, the SDK returns one of two kinds of backend:

- **Provider-specific backends** — Some hardware vendors (e.g. IonQ, Quantinuum) ship dedicated Qiskit backend classes that have native integration with their APIs. These handle gate translation and result parsing using hardware-specific logic.
- **Generic QIR backends** — For any other target that accepts QIR input, the SDK automatically wraps it as a generic backend. These compile the Qiskit circuit to OpenQASM 3 and then to QIR internally, so you don't have to manage those steps manually.

In practice `get_backend()` selects the right type for you — you use the same call regardless of which backend type is returned.

In [ ]:
from qdk.azure import Workspace
from qdk.azure.qiskit import AzureQuantumProvider

workspace = Workspace(
    subscription_id=subscription_id,
    resource_group=resource_group,
    name=workspace_name,
    location=location,
)

provider = AzureQuantumProvider(workspace)

# List available backends
for backend in provider.backends():
    print(f"{backend.name:45s}  {type(backend).__name__}")

In [ ]:
# Replace with any backend name from the list above
target_name = "rigetti.sim.qvm"
backend = provider.get_backend(target_name)

job = backend.run(circuit, shots=100, job_name="qiskit-provider-job")
print(f"Job {job.job_id()} submitted — waiting for results...")

counts = job.result().get_counts()
total = sum(counts.values())
print(f"\nResults ({total} shots):")
for bitstring, count in sorted(counts.items()):
    print(f"  {bitstring}: {count:4d}  ({count/total:.1%})")

## Parameterized Qiskit circuits

Many algorithms use symbolic parameters. Before submitting to azure (or before compiling to QIR), all circuit parameters must be bound to numeric values. Below we build a simple entangling ladder, apply a rotation parameterized by `θ` across all qubits, then uncompute the entanglement and measure the first qubit.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

def get_parameterized_circuit(n = 5) -> QuantumCircuit:
    theta = Parameter("θ")
    qc = QuantumCircuit(n, 1)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.rz(theta, range(n))

    for i in reversed(range(n - 1)):
        qc.cx(i, i + 1)
    qc.h(0)
    qc.measure(0, 0)
    return qc

# Build the symbolic (parameterized) circuit
parameterized_circuit = get_parameterized_circuit()

# Bind θ to a numeric value, then visualize the bound circuit
bound_circuit = parameterized_circuit.assign_parameters({"θ": 0.5})

bound_circuit.draw(output="text")

## Submitting a bound parameterized circuit

Here we submit the `bound_circuit` from above using the recommended `AzureQuantumProvider` approach. The printed result shows the measurement counts. Change the value of `θ` (or loop over several values) to explore how the outcome distribution varies.

In [ ]:
parameterized_job = backend.run(bound_circuit, shots=100, job_name="qiskit-parameterized-job")
print(f"Job {parameterized_job.job_id()} submitted — waiting for results...")

parameterized_counts = parameterized_job.result().get_counts()
print(parameterized_counts)

## Handling qubit loss on noisy hardware

On some hardware backends — particularly neutral-atom and trapped-ion devices — a qubit may be lost before measurement (e.g. an atom is ejected from the trap). When this happens, the backend records `"-"` in the bitstring position for that qubit rather than `"0"` or `"1"`. Because loss shots contain non-binary characters, they cannot be included in standard counts or probability distributions, which assume a fixed binary alphabet. The SDK therefore separates them automatically.

The result data (`job.result().results[0].data`) exposes two parallel sets of fields:

| Field | What it contains |
|---|---|
| **`counts`** | Bitstring → shot count, accepted shots only (no `"-"`) |
| **`probabilities`** | Bitstring → empirical probability, accepted shots only |
| **`memory`** | Per-shot bitstring list, accepted shots only |
| **`raw_counts`** | Bitstring → shot count, all shots (loss shots have `"-"`) |
| **`raw_probabilities`** | Bitstring → empirical probability, all shots |
| **`raw_memory`** | Per-shot bitstring list, all shots |

Use the **accepted** fields (`counts`, `probabilities`, `memory`) for any downstream analysis that expects clean binary bitstrings. Use the **raw** fields (`raw_counts`, `raw_probabilities`, `raw_memory`) to inspect loss patterns — for example, to calculate the overall loss rate or identify which qubit positions are being lost most frequently.

> **Tip**: A high loss rate may indicate hardware instability or a circuit that is too deep for the current calibration.